# Section 1: Geocode EVA Station IDs to Latitude/Longitude

## Objective

Extract all unique `eva` station IDs from the cleaned ICE data, geocode each station to `latitude`/`longitude`, and save the lookup table to disk for the weather merge.

## Why this step is important

Open-Meteo requires coordinates — not station names or `eva` codes. Without geocoding, weather cannot be joined to railway rows.

## What problem does it solve?

It answers: *What are the map coordinates for each of the 95 ICE stations in our dataset?*

## Methodology

1. Load `ice_cleaned_2024-07.parquet` from disk.
2. Extract unique `eva` + `xml_station_name` pairs.
3. Geocode via **Nominatim (OpenStreetMap)** — returns railway station coordinates.
   - Open-Meteo Geocoding API was tested but does not resolve `Hbf` station names reliably.
4. Query format: `"{station_name}, Germany"` with 1 req/sec (Nominatim policy).
5. Save `data/reference/station_coordinates.json` + geocoding report.

## Expected Output

- ~95 unique stations geocoded
- Success rate ≥ 90%
- Saved coordinate lookup: `eva → latitude, longitude`
- Failed stations logged for manual review

## Interpretation

- **`class: railway, type: station`** in response → correct match
- **Slight coord differences vs weather grid** → normal; Open-Meteo snaps to its grid
- **Failed geocoding** → station name too unusual; try manual fix in report

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| DB data has no coordinates | Nominatim geocoding by station name |
| 95 API calls | 1 req/sec + cache to JSON |
| Unusual station name formats | Fallback queries with cleaned names |

## Summary

Station coordinates mapped from `eva` IDs and saved to disk — ready for per-station weather fetch in Section 2.

## Next Step

**Section 2:** Fetch hourly Open-Meteo weather for all geocoded stations (July 2024).

---

**Key Takeaways**
- `eva` → `(latitude, longitude)` lookup created
- Nominatim used (not Open-Meteo geocoding) for German rail stations
- Output: `station_coordinates.json`

**What Comes Next**
- Section 2: fetch weather per station → merge with ICE data

In [1]:
# =============================================================================
# Notebook 04 | Section 1: Geocode EVA Station IDs to Latitude/Longitude
# =============================================================================
# Self-contained: loads ice_cleaned from disk; geocodes via Nominatim API.
# Saves station_coordinates.json for Section 2 weather fetch + merge.
# =============================================================================

from __future__ import annotations

import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_MONTH = "2024-07"
ICE_CLEANED = PROCESSED_DIR / f"ice_cleaned_{TARGET_MONTH}.parquet"
STATION_COORDS_PATH = REFERENCE_DIR / "station_coordinates.json"
GEOCODING_REPORT_PATH = REFERENCE_DIR / f"geocoding_report_{TARGET_MONTH}.json"

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
# Nominatim requires a valid User-Agent — replace with your name/email
USER_AGENT = "ICE-Train-Delay-Capstone/1.0 (university-ml-project)"
REQUEST_DELAY_SEC = 1.1   # Nominatim policy: max 1 request per second


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


# =============================================================================
# 2. LOAD CLEANED ICE DATA — extract unique stations
# =============================================================================
if not ICE_CLEANED.exists():
    raise FileNotFoundError(
        f"Missing: {ICE_CLEANED}\nRun Notebook 03 first."
    )

print(f"Loading: {ICE_CLEANED}")
ice_df = pd.read_parquet(ICE_CLEANED)

# One row per unique eva — use most frequent station name per eva
stations = (
    ice_df.groupby("eva")
    .agg(
        xml_station_name=("xml_station_name", lambda x: x.mode().iloc[0]),
        station_name=("station_name", lambda x: x.mode().iloc[0]),
        stop_count=("id", "count"),
    )
    .reset_index()
    .sort_values("stop_count", ascending=False)
)

print(f"Unique eva stations to geocode: {len(stations)}")
print()

# =============================================================================
# 3. GEOCODING FUNCTIONS
# =============================================================================
def clean_station_name(name: str) -> str:
    """Remove S-Bahn suffixes and extra whitespace from station names."""
    name = re.sub(r"\s*\(S\).*", "", name)       # remove (S-Bahn) suffix
    name = re.sub(r"\s*\(tief\).*", "", name)    # remove (tief) suffix
    name = re.sub(r"\s+", " ", name).strip()
    return name


def build_geocode_queries(xml_name: str, station_name: str) -> list[str]:
    """Build ordered list of query strings to try."""
    queries = []
    for name in [xml_name, station_name, clean_station_name(xml_name)]:
        if name and name not in queries:
            queries.append(f"{name}, Germany")
    return queries


def geocode_nominatim(
    query: str,
    user_agent: str = USER_AGENT,
    timeout: int = 15,
) -> dict | None:
    """
    Geocode a station name via Nominatim (OpenStreetMap).

    Returns dict with lat, lon, display_name, class, type — or None if not found.
    """
    headers = {"User-Agent": user_agent}
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "countrycodes": "de",
    }
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=headers,
        timeout=timeout,
    )
    response.raise_for_status()
    results = response.json()

    if not results:
        return None

    hit = results[0]
    return {
        "latitude": float(hit["lat"]),
        "longitude": float(hit["lon"]),
        "display_name": hit.get("display_name", ""),
        "osm_class": hit.get("class", ""),
        "osm_type": hit.get("type", ""),
        "query_used": query,
    }


def geocode_station(
    eva: str,
    xml_name: str,
    station_name: str,
) -> dict:
    """
    Try multiple query variants for one station.
    Returns result dict with status 'success' or 'failed'.
    """
    queries = build_geocode_queries(xml_name, station_name)

    for query in queries:
        try:
            result = geocode_nominatim(query)
            time.sleep(REQUEST_DELAY_SEC)   # respect Nominatim rate limit

            if result is None:
                continue

            # Prefer railway station matches
            is_railway = (
                result["osm_class"] == "railway"
                or "station" in result["osm_type"].lower()
                or "bahnhof" in result["display_name"].lower()
            )

            return {
                "eva": eva,
                "xml_station_name": xml_name,
                "station_name": station_name,
                "status": "success",
                "latitude": result["latitude"],
                "longitude": result["longitude"],
                "display_name": result["display_name"],
                "osm_class": result["osm_class"],
                "osm_type": result["osm_type"],
                "query_used": result["query_used"],
                "is_railway_match": is_railway,
            }

        except Exception as exc:
            time.sleep(REQUEST_DELAY_SEC)
            return {
                "eva": eva,
                "xml_station_name": xml_name,
                "status": "failed",
                "error": str(exc),
                "queries_tried": queries,
            }

    return {
        "eva": eva,
        "xml_station_name": xml_name,
        "station_name": station_name,
        "status": "failed",
        "error": "No results from Nominatim for any query variant",
        "queries_tried": queries,
    }


# =============================================================================
# 4. GEOCODE ALL STATIONS
# =============================================================================
print("Geocoding stations via Nominatim (1 req/sec — ~2 min for 95 stations)...")
print()

geocoding_results: list[dict] = []

for i, row in stations.iterrows():
    eva = row["eva"]
    xml_name = row["xml_station_name"]
    station_name = row["station_name"]
    stop_count = row["stop_count"]

    print(f"  [{len(geocoding_results)+1}/{len(stations)}] eva={eva}  {xml_name[:40]}")

    result = geocode_station(eva, xml_name, station_name)
    result["stop_count"] = int(stop_count)
    geocoding_results.append(result)

    if result["status"] == "success":
        print(f"         → ({result['latitude']:.4f}, {result['longitude']:.4f})  [{result['query_used'][:50]}]")
    else:
        print(f"         → FAILED: {result.get('error', 'unknown')}")

print()

# =============================================================================
# 5. BUILD COORDINATE LOOKUP TABLE
# =============================================================================
successful = [r for r in geocoding_results if r["status"] == "success"]
failed = [r for r in geocoding_results if r["status"] == "failed"]

success_rate = round(100 * len(successful) / len(geocoding_results), 1)

# eva → coordinates lookup (used in Section 2 merge)
station_coordinates = {
    r["eva"]: {
        "latitude": r["latitude"],
        "longitude": r["longitude"],
        "xml_station_name": r["xml_station_name"],
        "station_name": r.get("station_name", ""),
        "display_name": r["display_name"],
        "query_used": r["query_used"],
        "stop_count": r["stop_count"],
    }
    for r in successful
}

station_coords_artifact = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "geocoding_service": "Nominatim (OpenStreetMap)",
        "geocoding_url": NOMINATIM_URL,
        "total_stations": len(stations),
        "successful": len(successful),
        "failed": len(failed),
        "success_rate_pct": success_rate,
    },
    "stations": station_coordinates,
    "failed_stations": failed,
}

with STATION_COORDS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(station_coords_artifact), f, indent=2, ensure_ascii=False)

geocoding_report = {
    "metadata": station_coords_artifact["metadata"],
    "results": geocoding_results,
    "success_rate_pct": success_rate,
    "ready_for_section_2": success_rate >= 90.0,
}

with GEOCODING_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(geocoding_report), f, indent=2, ensure_ascii=False)

# =============================================================================
# 6. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 1: Geocode EVA Station IDs to Latitude/Longitude")
print(sep)
print(f"Total stations  : {len(stations)}")
print(f"Geocoded OK     : {len(successful)}  ({success_rate}%)")
print(f"Failed          : {len(failed)}")
print()

if successful:
    print("--- SAMPLE GEOCODED STATIONS (top 5 by stop count) ---")
    sample = sorted(successful, key=lambda x: x["stop_count"], reverse=True)[:5]
    for s in sample:
        print(
            f"  eva={s['eva']}  {s['xml_station_name'][:30]:<30} "
            f"→ ({s['latitude']:.4f}, {s['longitude']:.4f})"
        )
    print()

if failed:
    print("--- FAILED STATIONS (manual review needed) ---")
    for f_station in failed:
        print(f"  eva={f_station['eva']}  {f_station.get('xml_station_name', '')}")
        print(f"    queries tried: {f_station.get('queries_tried', [])}")
    print()

print(f"--- SAVED ---")
print(f"  Coordinates : {STATION_COORDS_PATH}")
print(f"  Report      : {GEOCODING_REPORT_PATH}")
print()

if success_rate >= 90:
    print("Geocoding complete. Ready for Section 2: fetch weather per station.")
else:
    print(f"WARNING: Success rate {success_rate}% < 90%. Review failed stations before Section 2.")

print(sep)

Loading: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_cleaned_2024-07.parquet
Unique eva stations to geocode: 95

Geocoding stations via Nominatim (1 req/sec — ~2 min for 95 stations)...

  [1/95] eva=08000105  Frankfurt(Main)Hbf
         → (50.1067, 8.6626)  [Frankfurt(Main)Hbf, Germany]
  [2/95] eva=08000261  München Hbf
         → (48.1407, 11.5569)  [München Hbf, Germany]
  [3/95] eva=08000152  Hannover Hbf
         → (52.3771, 9.7418)  [Hannover Hbf, Germany]
  [4/95] eva=08010404  Berlin-Spandau
         → (52.5348, 13.1956)  [Berlin-Spandau, Germany]
  [5/95] eva=08002549  Hamburg Hbf
         → (53.5532, 10.0064)  [Hamburg Hbf, Germany]
  [6/95] eva=08000207  Köln Hbf
         → (50.9428, 6.9591)  [Köln Hbf, Germany]
  [7/95] eva=08000284  Nürnberg Hbf
         → (49.4462, 11.0819)  [Nürnberg Hbf, Germany]
  [8/95] eva=08003200  Kassel-Wilhelmshöhe
         → (51.3114, 9.4477)  [Kassel-Wilhelmshöhe, Germany]
  [9/95] eva=08098160  Berlin Hbf
         → (52

# Section 2: Fetch Hourly Weather for All Geocoded Stations

## Objective

For each geocoded `eva` station, fetch hourly Open-Meteo weather for July 2024, combine into one DataFrame, and save to disk for the merge in Section 3.

## Why this step is important

Section 1 gave us coordinates. This section downloads the actual weather values that will be joined to each ICE stop.

## What problem does it solve?

It answers: *What was the temperature, rain, wind, etc. at each station for every hour in July 2024?*

## Methodology

1. Load `station_coordinates.json` and `ice_cleaned_2024-07.parquet` from disk.
2. Get date range from ICE data (`2024-07-01` → `2024-07-31`).
3. For each of the ~95 stations, call Open-Meteo Archive API with its `latitude`/`longitude`.
4. Tag each weather row with `eva` + coordinates.
5. Combine all stations → save `weather_by_station_2024-07.parquet`.
6. Cache progress after each station (resumable if interrupted).

## Expected Output

- ~95 stations × 744 hours ≈ **70,000+ weather rows**
- Saved parquet: `weather_by_station_2024-07.parquet`
- Fetch report with success/fail per station

## Interpretation

- **744 hours per station** = 31 days × 24 hours (full July)
- **Rows = stations × hours** → one weather row per station per hour
- **Failed fetches** → logged; station excluded from merge
- **API returns grid-snapped coords** → slight difference from Nominatim is normal

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| 95 API calls | Sequential fetch with progress + cache |
| Interrupted download | Resume from partial cache file |
| Large combined file | Parquet compression on disk |

## Summary

Hourly weather for all geocoded stations saved to disk — ready for left join in Section 3.

## Next Step

**Section 3:** Merge ICE railway rows with weather on `eva` + `departure_planned_hour_naive`.

---

**Key Takeaways**
- One combined weather file for all stations
- Each row tagged with `eva` for easy join
- Output: `weather_by_station_2024-07.parquet`

**What Comes Next**
- Section 3: left join + merge validation

In [2]:
# =============================================================================
# Notebook 04 | Section 2: Fetch Hourly Weather for All Geocoded Stations
# =============================================================================
# Self-contained: loads station_coordinates.json + ice_cleaned from disk.
# Fetches Open-Meteo hourly weather per station; saves combined parquet.
# Resumable: skips stations already in cache file if re-run.
# =============================================================================

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# =============================================================================
# 1. PATHS & SETTINGS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_MONTH = "2024-07"
ICE_CLEANED = PROCESSED_DIR / f"ice_cleaned_{TARGET_MONTH}.parquet"
STATION_COORDS_PATH = REFERENCE_DIR / "station_coordinates.json"
WEATHER_OUTPUT = PROCESSED_DIR / f"weather_by_station_{TARGET_MONTH}.parquet"
WEATHER_CACHE = PROCESSED_DIR / f"weather_by_station_{TARGET_MONTH}_cache.parquet"
WEATHER_FETCH_REPORT = REFERENCE_DIR / f"weather_fetch_report_{TARGET_MONTH}.json"

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_TIMEOUT = 60
REQUEST_DELAY_SEC = 0.3   # polite pause between API calls
DEMO_TIMEZONE = "Europe/Berlin"

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


# =============================================================================
# 2. LOAD STATION COORDINATES + ICE DATE RANGE
# =============================================================================
if not STATION_COORDS_PATH.exists():
    raise FileNotFoundError(
        f"Missing: {STATION_COORDS_PATH}\nRun Notebook 04 Section 1 first."
    )
if not ICE_CLEANED.exists():
    raise FileNotFoundError(
        f"Missing: {ICE_CLEANED}\nRun Notebook 03 first."
    )

coords_data = load_json(STATION_COORDS_PATH)
stations = coords_data["stations"]   # eva → {latitude, longitude, ...}

ice_df = pd.read_parquet(ICE_CLEANED, columns=["time"])
start_date = str(ice_df["time"].min().date())
end_date = str(ice_df["time"].max().date())

print(f"Stations to fetch : {len(stations)}")
print(f"Date range        : {start_date} → {end_date}")
print(f"Hourly variables  : {HOURLY_VARIABLES}")
print()

# =============================================================================
# 3. OPEN-METEO FETCH FUNCTION
# =============================================================================
def fetch_weather_for_station(
    eva: str,
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    hourly_variables: list[str],
    timezone: str = DEMO_TIMEZONE,
) -> pd.DataFrame:
    """
    Fetch hourly weather for one station from Open-Meteo Archive API.
    Returns DataFrame with eva, lat, lon, time, and weather columns.
    """
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone,
    }
    response = requests.get(
        OPEN_METEO_ARCHIVE_URL,
        params=params,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()

    hourly = data["hourly"]
    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])

    # Tag with station identifiers for merge in Section 3
    df["eva"] = eva
    df["latitude"] = data.get("latitude", latitude)
    df["longitude"] = data.get("longitude", longitude)
    df["timezone"] = data.get("timezone", timezone)

    return df


# =============================================================================
# 4. RESUMABLE FETCH — skip stations already in cache
# =============================================================================
already_fetched: set[str] = set()
cached_frames: list[pd.DataFrame] = []

if WEATHER_CACHE.exists():
    cache_df = pd.read_parquet(WEATHER_CACHE)
    already_fetched = set(cache_df["eva"].unique())
    cached_frames.append(cache_df)
    print(f"Resuming: {len(already_fetched)} stations already in cache.")
    print()

fetch_log: list[dict] = []
new_frames: list[pd.DataFrame] = []

stations_to_fetch = {
    eva: info for eva, info in stations.items()
    if eva not in already_fetched
}

print(f"Fetching weather for {len(stations_to_fetch)} stations...")
print("(Requires internet — ~1–2 minutes)")
print()

for i, (eva, info) in enumerate(stations_to_fetch.items(), start=1):
    station_label = info.get("xml_station_name", eva)[:40]
    print(f"  [{i}/{len(stations_to_fetch)}] eva={eva}  {station_label}")

    try:
        wx_df = fetch_weather_for_station(
            eva=eva,
            latitude=info["latitude"],
            longitude=info["longitude"],
            start_date=start_date,
            end_date=end_date,
            hourly_variables=HOURLY_VARIABLES,
        )
        new_frames.append(wx_df)

        fetch_log.append({
            "eva": eva,
            "status": "success",
            "rows": len(wx_df),
            "latitude": info["latitude"],
            "longitude": info["longitude"],
        })
        print(f"         → {len(wx_df)} hourly rows fetched")

        # Save cache after each station (resumable)
        all_so_far = pd.concat(cached_frames + new_frames, ignore_index=True)
        all_so_far.to_parquet(WEATHER_CACHE, index=False)

    except Exception as exc:
        fetch_log.append({
            "eva": eva,
            "status": "failed",
            "error": str(exc),
        })
        print(f"         → FAILED: {exc}")

    time.sleep(REQUEST_DELAY_SEC)

print()

# =============================================================================
# 5. COMBINE ALL STATION WEATHER DATA
# =============================================================================
all_frames = cached_frames + new_frames

if not all_frames:
    raise ValueError("No weather data fetched. Check internet and station coordinates.")

weather_all = pd.concat(all_frames, ignore_index=True)

# Validate structure
expected_cols = ["eva", "time", "latitude", "longitude"] + HOURLY_VARIABLES
missing_cols = [c for c in expected_cols if c not in weather_all.columns]
if missing_cols:
    raise ValueError(f"Weather DataFrame missing columns: {missing_cols}")

# =============================================================================
# 6. WEATHER DATA SUMMARY
# =============================================================================
weather_summary = {
    "total_rows": int(len(weather_all)),
    "unique_stations": int(weather_all["eva"].nunique()),
    "unique_hours": int(weather_all["time"].nunique()),
    "rows_per_station_avg": round(len(weather_all) / weather_all["eva"].nunique(), 1),
    "date_min": str(weather_all["time"].min()),
    "date_max": str(weather_all["time"].max()),
    "temperature_2m_mean": round(float(weather_all["temperature_2m"].mean()), 2),
    "precipitation_total_mm": round(float(weather_all["precipitation"].sum()), 2),
    "hours_with_rain": int((weather_all["precipitation"] > 0).sum()),
}

successful_fetches = [f for f in fetch_log if f["status"] == "success"]
failed_fetches = [f for f in fetch_log if f["status"] == "failed"]

# =============================================================================
# 7. SAVE FINAL WEATHER FILE + REPORT
# =============================================================================
weather_all.to_parquet(WEATHER_OUTPUT, index=False)

fetch_report = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "api_url": OPEN_METEO_ARCHIVE_URL,
        "date_range": {"start": start_date, "end": end_date},
        "timezone": DEMO_TIMEZONE,
        "hourly_variables": HOURLY_VARIABLES,
    },
    "fetch_summary": {
        "stations_in_coords_file": len(stations),
        "stations_fetched_this_run": len(stations_to_fetch),
        "stations_from_cache": len(already_fetched),
        "successful": len(successful_fetches),
        "failed": len(failed_fetches),
        "total_weather_rows": weather_summary["total_rows"],
    },
    "weather_summary": weather_summary,
    "fetch_log": fetch_log,
    "output_files": {
        "weather_parquet": str(WEATHER_OUTPUT),
        "weather_cache": str(WEATHER_CACHE),
    },
    "ready_for_section_3": len(failed_fetches) == 0,
}

with WEATHER_FETCH_REPORT.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(fetch_report), f, indent=2, ensure_ascii=False)

# =============================================================================
# 8. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 2: Fetch Hourly Weather for All Geocoded Stations")
print(sep)
print(f"Date range         : {start_date} → {end_date}")
print(f"Stations in file   : {weather_summary['unique_stations']}")
print(f"Total weather rows : {weather_summary['total_rows']:,}")
print(f"Avg rows/station   : {weather_summary['rows_per_station_avg']}")
print(f"Unique hours       : {weather_summary['unique_hours']}")
print()

print("--- WEATHER SUMMARY ---")
print(f"  Mean temperature : {weather_summary['temperature_2m_mean']} °C")
print(f"  Total precipitation: {weather_summary['precipitation_total_mm']} mm")
print(f"  Hours with rain  : {weather_summary['hours_with_rain']:,}")
print()

if failed_fetches:
    print(f"--- FAILED FETCHES ({len(failed_fetches)}) ---")
    for f in failed_fetches:
        print(f"  eva={f['eva']}: {f.get('error', '')}")
    print()

print("--- SAMPLE WEATHER ROWS ---")
sample_cols = ["eva", "time", "temperature_2m", "precipitation", "windspeed_10m"]
print(weather_all[sample_cols].head(3).to_string(index=False))
print()

print(f"--- SAVED ---")
print(f"  Weather data : {WEATHER_OUTPUT}")
print(f"  Fetch report : {WEATHER_FETCH_REPORT}")
print()
print("Later notebooks load with:")
print(f'  pd.read_parquet("{WEATHER_OUTPUT}")')
print()
print("Ready for Section 3: merge ICE data with weather.")
print(sep)

Stations to fetch : 95
Date range        : 2024-07-01 → 2024-07-31
Hourly variables  : ['temperature_2m', 'precipitation', 'rain', 'snowfall', 'windspeed_10m', 'windgusts_10m', 'weather_code', 'visibility']

Fetching weather for 95 stations...
(Requires internet — ~1–2 minutes)

  [1/95] eva=08000105  Frankfurt(Main)Hbf
         → 744 hourly rows fetched
  [2/95] eva=08000261  München Hbf
         → 744 hourly rows fetched
  [3/95] eva=08000152  Hannover Hbf
         → 744 hourly rows fetched
  [4/95] eva=08010404  Berlin-Spandau
         → 744 hourly rows fetched
  [5/95] eva=08002549  Hamburg Hbf
         → 744 hourly rows fetched
  [6/95] eva=08000207  Köln Hbf
         → 744 hourly rows fetched
  [7/95] eva=08000284  Nürnberg Hbf
         → 744 hourly rows fetched
  [8/95] eva=08003200  Kassel-Wilhelmshöhe
         → 744 hourly rows fetched
  [9/95] eva=08098160  Berlin Hbf
         → 744 hourly rows fetched
  [10/95] eva=08000128  Göttingen
         → 744 hourly rows fetched
  [11

# Section 3: Merge ICE Railway Data with Weather & Validate

## Objective

Left-join cleaned ICE stops with hourly weather on `eva` + `departure_planned_hour`, run validation checks, and save the merged dataset to disk.

## Why this step is important

This is the core data-integration step — it combines operational and weather data into one modeling-ready table.

## What problem does it solve?

It answers: *Does each ICE stop have the correct weather for its station and planned departure hour?*

## Methodology

**Merge keys:**

| Railway | Weather |
|---------|---------|
| `eva` | `eva` |
| `departure_planned_hour_naive` | `time` |

**Join type:** Left join — keep all ICE rows; weather null where no match.

**Validations (from `merge_strategy.json`):**
- Merge rate ≥ 90% on mergeable rows
- Row count unchanged after join
- No duplicate `id` values
- Weather nulls not systematic by station

Save `ice_weather_merged_2024-07.parquet` + validation report.

## Expected Output

- Merged row count = ICE cleaned row count (146,389)
- Merge rate ≥ 90% on mergeable rows
- Weather columns appended to railway columns
- Validation report saved

## Interpretation

- **Merge rate ~100% on mergeable rows** → join keys work correctly
- **Non-mergeable rows (~9.65%)** → no departure time; weather stays null (expected)
- **New columns** = 8 weather variables added to each matched row
- **This file** → input for EDA (Notebook 05) and modeling

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Two-key join (station + hour) | `eva` + `departure_planned_hour_naive` |
| Non-mergeable rows | Left join; null weather expected |
| Duplicate join keys | Validate uniqueness of weather `eva`+`time` |

## Summary

ICE operational data and weather are merged and validated. Integrated dataset saved for downstream analysis.

## Next Step

**Section 4:** Notebook 04 close-out — summarize integration, list all files, confirm readiness for Notebook 05 (EDA).

---

**Key Takeaways**
- Join: same station (`eva`) + same hour (planned departure)
- Left join preserves all ICE rows
- Output: `ice_weather_merged_2024-07.parquet`

**What Comes Next**
- Notebook 05: exploratory data analysis on merged data

In [3]:
# =============================================================================
# Notebook 04 | Section 3: Merge ICE Railway Data with Weather & Validate
# =============================================================================
# Self-contained: loads ice_cleaned + weather_by_station from disk.
# Left join on eva + departure_planned_hour_naive == time.
# Saves ice_weather_merged_2024-07.parquet + validation report.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_MONTH = "2024-07"
ICE_CLEANED = PROCESSED_DIR / f"ice_cleaned_{TARGET_MONTH}.parquet"
WEATHER_PARQUET = PROCESSED_DIR / f"weather_by_station_{TARGET_MONTH}.parquet"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"
MERGED_OUTPUT = PROCESSED_DIR / f"ice_weather_merged_{TARGET_MONTH}.parquet"
MERGE_VALIDATION_PATH = REFERENCE_DIR / f"merge_validation_report_{TARGET_MONTH}.json"

WEATHER_COLS = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


# =============================================================================
# 2. LOAD DATA FROM DISK
# =============================================================================
for path, label in [
    (ICE_CLEANED, "ice_cleaned"),
    (WEATHER_PARQUET, "weather_by_station"),
    (MERGE_STRATEGY_PATH, "merge_strategy"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

print(f"Loading ICE data   : {ICE_CLEANED}")
ice_df = pd.read_parquet(ICE_CLEANED)
rows_before = len(ice_df)

print(f"Loading weather    : {WEATHER_PARQUET}")
weather_df = pd.read_parquet(WEATHER_PARQUET)

merge_strategy = load_json(MERGE_STRATEGY_PATH)
print(f"ICE rows           : {rows_before:,}")
print(f"Weather rows       : {len(weather_df):,}")
print()

# =============================================================================
# 3. PREPARE WEATHER TABLE FOR JOIN
# =============================================================================
# Keep only columns needed for merge + weather features
weather_join = weather_df[
    ["eva", "time", "latitude", "longitude"] + WEATHER_COLS
].copy()

# Rename to avoid column clashes after join
weather_join = weather_join.rename(columns={
    "time": "weather_time",
    "latitude": "weather_latitude",
    "longitude": "weather_longitude",
})

# Ensure weather_time is datetime (naive, Europe/Berlin from API)
weather_join["weather_time"] = pd.to_datetime(weather_join["weather_time"])

# Check weather key uniqueness (eva + hour must be unique)
weather_dupes = weather_join.duplicated(subset=["eva", "weather_time"]).sum()
if weather_dupes > 0:
    print(f"WARNING: {weather_dupes} duplicate eva+time in weather — keeping first.")
    weather_join = weather_join.drop_duplicates(
        subset=["eva", "weather_time"], keep="first"
    )

print(f"Weather join keys  : eva + weather_time")
print(f"Unique eva in wx   : {weather_join['eva'].nunique()}")
print(f"Duplicate keys     : {weather_dupes}")
print()

# =============================================================================
# 4. LEFT JOIN — railway ON weather
# =============================================================================
print("Merging on:")
print("  railway.eva                        == weather.eva")
print("  railway.departure_planned_hour_naive == weather.weather_time")
print()

merged_df = ice_df.merge(
    weather_join,
    left_on=["eva", "departure_planned_hour_naive"],
    right_on=["eva", "weather_time"],
    how="left",
)

rows_after = len(merged_df)

# =============================================================================
# 5. MERGE RATE CALCULATION
# =============================================================================
# A row is "weather-matched" if temperature_2m is not null
merged_df["weather_matched"] = merged_df["temperature_2m"].notna()

total_matched = int(merged_df["weather_matched"].sum())
overall_merge_rate = round(100 * total_matched / rows_after, 2)

# Merge rate on mergeable rows only (rows with departure time)
if "mergeable" in merged_df.columns:
    mergeable_df = merged_df[merged_df["mergeable"] == True]  # noqa: E712
    mergeable_matched = int(mergeable_df["weather_matched"].sum())
    mergeable_merge_rate = round(
        100 * mergeable_matched / len(mergeable_df), 2
    )
    non_mergeable_count = int((~merged_df["mergeable"]).sum())
else:
    mergeable_df = merged_df[merged_df["departure_planned_hour_naive"].notna()]
    mergeable_matched = int(mergeable_df["weather_matched"].sum())
    mergeable_merge_rate = round(
        100 * mergeable_matched / len(mergeable_df), 2
    )
    non_mergeable_count = 0

# Stations with systematically missing weather
station_miss = (
    merged_df[merged_df["mergeable"] == True]  # noqa: E712
    .groupby("eva")
    .agg(
        total=("id", "count"),
        matched=("weather_matched", "sum"),
    )
)
station_miss["miss_rate_pct"] = round(
    100 * (1 - station_miss["matched"] / station_miss["total"]), 2
)
problem_stations = station_miss[station_miss["miss_rate_pct"] > 50]

# =============================================================================
# 6. VALIDATION CHECKS (from merge_strategy.json)
# =============================================================================
validations = {
    "row_count_preserved": {
        "pass": bool(rows_after == rows_before),
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rule": "Left join must not change row count",
    },
    "merge_rate_on_mergeable": {
        "pass": bool(mergeable_merge_rate >= 90.0),
        "merge_rate_pct": mergeable_merge_rate,
        "threshold_pct": 90.0,
        "mergeable_rows": int(len(mergeable_df)),
        "matched_rows": mergeable_matched,
        "rule": "≥ 90% of mergeable rows must have weather",
    },
    "no_duplicate_ids": {
        "pass": bool(not merged_df["id"].duplicated().any()),
        "duplicate_count": int(merged_df["id"].duplicated().sum()),
        "rule": "id must be unique after merge",
    },
    "weather_columns_present": {
        "pass": bool(all(c in merged_df.columns for c in WEATHER_COLS)),
        "columns": WEATHER_COLS,
        "rule": "All 8 weather variables must be in merged data",
    },
    "no_systematic_station_gaps": {
        "pass": bool(len(problem_stations) == 0),
        "stations_with_high_miss_rate": int(len(problem_stations)),
        "rule": "No station should have >50% missing weather among mergeable rows",
    },
}

failed = [k for k, v in validations.items() if not v["pass"]]
validation_passed = len(failed) == 0

# =============================================================================
# 7. MERGED DATASET SUMMARY
# =============================================================================
merged_summary = {
    "rows": int(len(merged_df)),
    "columns": int(len(merged_df.columns)),
    "overall_merge_rate_pct": overall_merge_rate,
    "mergeable_merge_rate_pct": mergeable_merge_rate,
    "non_mergeable_rows": non_mergeable_count,
    "weather_matched_rows": total_matched,
    "unique_eva_stations": int(merged_df["eva"].nunique()),
    "delay_mean_min": round(float(merged_df["delay_in_min"].mean()), 2),
    "weather_temperature_mean": round(
        float(merged_df["temperature_2m"].mean()), 2
    ),
    "new_weather_columns": WEATHER_COLS,
}

# =============================================================================
# 8. SAVE MERGED DATA + VALIDATION REPORT
# =============================================================================
merged_df.to_parquet(MERGED_OUTPUT, index=False)

validation_report = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "join_type": "left",
        "join_keys": {
            "railway": ["eva", "departure_planned_hour_naive"],
            "weather": ["eva", "weather_time"],
        },
    },
    "merge_summary": merged_summary,
    "validations": validations,
    "validation_passed": validation_passed,
    "failed_checks": failed,
    "problem_stations": (
        problem_stations.reset_index().to_dict(orient="records")
        if len(problem_stations) > 0 else []
    ),
    "output_file": str(MERGED_OUTPUT),
    "ready_for_notebook_05": validation_passed,
}

with MERGE_VALIDATION_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(validation_report), f, indent=2, ensure_ascii=False)

# =============================================================================
# 9. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 3: Merge ICE Railway Data with Weather & Validate")
print(sep)
print(f"Rows before merge : {rows_before:,}")
print(f"Rows after merge  : {rows_after:,}")
print(f"Row count preserved: {'YES' if rows_after == rows_before else 'NO'}")
print()

print("--- MERGE RATES ---")
print(f"  Overall matched      : {total_matched:,} / {rows_after:,}  ({overall_merge_rate}%)")
print(f"  Mergeable rows       : {len(mergeable_df):,}")
print(f"  Mergeable matched    : {mergeable_matched:,}  ({mergeable_merge_rate}%)")
print(f"  Non-mergeable rows   : {non_mergeable_count:,}  (no departure time)")
print()

print("--- VALIDATIONS ---")
for key, val in validations.items():
    status = "PASS" if val["pass"] else "FAIL"
    print(f"  [{status}] {key}")
    if not val["pass"]:
        print(f"         → {val.get('rule', '')}")
print()

if len(problem_stations) > 0:
    print(f"--- STATIONS WITH HIGH MISS RATE ({len(problem_stations)}) ---")
    print(problem_stations.head(5).to_string())
    print()

print("--- SAMPLE MERGED ROWS ---")
sample_cols = [
    "eva", "xml_station_name", "departure_planned_hour_naive",
    "delay_in_min", "temperature_2m", "precipitation", "weather_matched",
]
print(merged_df[sample_cols].head(3).to_string(index=False))
print()

print(f"--- SAVED ---")
print(f"  Merged data        : {MERGED_OUTPUT}")
print(f"  Validation report: {MERGE_VALIDATION_PATH}")
print()

if validation_passed:
    print("All validations PASSED. Ready for Section 4: Notebook 04 close-out.")
else:
    print(f"WARNING: {len(failed)} validation(s) failed: {failed}")
    print("Review merge_validation_report before proceeding to EDA.")

print(sep)

Loading ICE data   : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_cleaned_2024-07.parquet
Loading weather    : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\weather_by_station_2024-07.parquet
ICE rows           : 146,389
Weather rows       : 70,680

Weather join keys  : eva + weather_time
Unique eva in wx   : 95
Duplicate keys     : 0

Merging on:
  railway.eva                        == weather.eva
  railway.departure_planned_hour_naive == weather.weather_time

Section 3: Merge ICE Railway Data with Weather & Validate
Rows before merge : 146,389
Rows after merge  : 146,389
Row count preserved: YES

--- MERGE RATES ---
  Overall matched      : 132,268 / 146,389  (90.35%)
  Mergeable rows       : 132,268
  Mergeable matched    : 132,268  (100.0%)
  Non-mergeable rows   : 14,121  (no departure time)

--- VALIDATIONS ---
  [PASS] row_count_preserved
  [PASS] merge_rate_on_mergeable
  [PASS] no_duplicate_ids
  [PASS] weather_columns_present
  [PASS] no

# Section 4: Integration Summary & Notebook 04 Close-Out

## Objective

Verify all Notebook 04 files exist, summarize the full integration pipeline, and confirm readiness for Notebook 05 (EDA).

## Why this step is important

This close-out proves the merge is complete, validated, and reproducible from saved files — not from notebook memory.

## What problem does it solve?

It answers: *Is the integrated ICE + weather dataset ready for analysis and modeling?*

## Methodology

Load all Notebook 04 artifacts, cross-check merge validation report, print pipeline recap, save `notebook_04_summary.json`.

## Expected Output

- All files marked OK
- Merge rate and validation status
- `notebook_04_summary.json` saved
- Ready for Notebook 05: YES

## Interpretation

- **Geocoded → fetched → merged** = complete data-integration chain
- **Merge rate ≥ 90%** on mergeable rows → join keys work
- **`ice_weather_merged_2024-07.parquet`** = main dataset for EDA and ML

## Challenges Faced

| Challenge | Resolution |
|-----------|------------|
| No coordinates in railway data | Nominatim geocoding (Section 1) |
| 95 API weather calls | Cached parquet per station |
| Two-key join | `eva` + `departure_planned_hour_naive` |

## Summary

Notebook 04 complete. Railway and weather data are integrated and validated.

## Next Step

**Notebook 05, Section 1:** Exploratory Data Analysis on the merged dataset.

---

### Notebook 04 — Closing

**Key Takeaways**
- 95 stations geocoded → weather fetched → left-joined to 146,389 ICE rows
- 8 weather features added to operational data
- Output: `ice_weather_merged_2024-07.parquet`

**What Comes Next**
- Notebook 05: delay distributions, weather-delay relationships, visualizations

In [4]:
# =============================================================================
# Notebook 04 | Section 4: Integration Summary & Notebook 04 Close-Out
# =============================================================================
# Self-contained: verifies all Notebook 04 artifacts and saves summary JSON.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_MONTH = "2024-07"
SUMMARY_PATH = REFERENCE_DIR / "notebook_04_summary.json"

ARTIFACTS = {
    "station_coordinates": REFERENCE_DIR / "station_coordinates.json",
    "geocoding_report": REFERENCE_DIR / f"geocoding_report_{TARGET_MONTH}.json",
    "weather_by_station": PROCESSED_DIR / f"weather_by_station_{TARGET_MONTH}.parquet",
    "weather_fetch_report": REFERENCE_DIR / f"weather_fetch_report_{TARGET_MONTH}.json",
    "ice_weather_merged": PROCESSED_DIR / f"ice_weather_merged_{TARGET_MONTH}.parquet",
    "merge_validation_report": REFERENCE_DIR / f"merge_validation_report_{TARGET_MONTH}.json",
}


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def check_file(path: Path) -> dict:
    if path.exists():
        return {
            "status": "OK",
            "path": str(path),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
        }
    return {"status": "MISSING", "path": str(path), "size_mb": None}


# =============================================================================
# 2. VERIFY ALL FILES
# =============================================================================
print("--- NOTEBOOK 04 ARTIFACT CHECK ---")
artifact_status = {name: check_file(path) for name, path in ARTIFACTS.items()}
for name, info in artifact_status.items():
    size = f"({info['size_mb']} MB)" if info["size_mb"] is not None else ""
    print(f"  [{info['status']}] {name} {size}")

missing = [n for n, i in artifact_status.items() if i["status"] == "MISSING"]
if missing:
    raise FileNotFoundError(
        f"Missing Notebook 04 files: {missing}\nComplete Sections 1–3 first."
    )

# =============================================================================
# 3. LOAD REPORTS + MERGED DATA
# =============================================================================
geocode_report = load_json(ARTIFACTS["geocoding_report"])
weather_report = load_json(ARTIFACTS["weather_fetch_report"])
merge_report = load_json(ARTIFACTS["merge_validation_report"])
merged_df = pd.read_parquet(ARTIFACTS["ice_weather_merged"])

# =============================================================================
# 4. BUILD SUMMARY
# =============================================================================
notebook_summary = {
    "metadata": {
        "notebook": "04_Data_Integration",
        "section": "Section 4 — Close-Out",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "status": "COMPLETE",
    },
    "pipeline_steps": [
        {
            "section": "Section 1",
            "action": "Geocode eva → latitude/longitude (Nominatim)",
            "stations_geocoded": geocode_report["metadata"]["successful"],
            "success_rate_pct": geocode_report["metadata"]["success_rate_pct"],
            "output": str(ARTIFACTS["station_coordinates"]),
        },
        {
            "section": "Section 2",
            "action": "Fetch hourly Open-Meteo weather per station",
            "weather_rows": weather_report["weather_summary"]["total_rows"],
            "stations": weather_report["weather_summary"]["unique_stations"],
            "output": str(ARTIFACTS["weather_by_station"]),
        },
        {
            "section": "Section 3",
            "action": "Left join ICE + weather on eva + hour",
            "merged_rows": merge_report["merge_summary"]["rows"],
            "mergeable_merge_rate_pct": merge_report["merge_summary"]["mergeable_merge_rate_pct"],
            "validation_passed": merge_report["validation_passed"],
            "output": str(ARTIFACTS["ice_weather_merged"]),
        },
    ],
    "merged_dataset": {
        "rows": int(len(merged_df)),
        "columns": int(len(merged_df.columns)),
        "unique_stations": int(merged_df["eva"].nunique()),
        "weather_matched_rows": int(merged_df["weather_matched"].sum())
        if "weather_matched" in merged_df.columns
        else int(merged_df["temperature_2m"].notna().sum()),
        "delay_mean_min": round(float(merged_df["delay_in_min"].mean()), 2),
        "weather_cols": [
            "temperature_2m", "precipitation", "rain", "snowfall",
            "windspeed_10m", "windgusts_10m", "weather_code", "visibility",
        ],
        "parquet_file": str(ARTIFACTS["ice_weather_merged"]),
    },
    "merge_validation": {
        "passed": merge_report["validation_passed"],
        "mergeable_merge_rate_pct": merge_report["merge_summary"]["mergeable_merge_rate_pct"],
        "failed_checks": merge_report.get("failed_checks", []),
    },
    "artifacts": artifact_status,
    "ready_for_notebook_05": merge_report["validation_passed"],
    "notebook_05_tasks": [
        "Load ice_weather_merged_2024-07.parquet",
        "Delay distribution analysis",
        "Weather vs delay relationship plots",
        "Class imbalance check for classification task",
        "Correlation heatmap",
    ],
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(notebook_summary), f, indent=2, ensure_ascii=False)

# =============================================================================
# 5. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print()
print(sep)
print("Notebook 04 — Integration Summary")
print(sep)

geo = geocode_report["metadata"]
wx = weather_report["weather_summary"]
mg = merge_report["merge_summary"]

print()
print("--- PIPELINE ---")
print(f"  Section 1 — Geocoded  : {geo['successful']} stations  ({geo['success_rate_pct']}%)")
print(f"  Section 2 — Weather   : {wx['total_rows']:,} rows  ({wx['unique_stations']} stations)")
print(f"  Section 3 — Merged    : {mg['rows']:,} rows  (merge rate {mg['mergeable_merge_rate_pct']}%)")
print()

print("--- MERGED DATASET ---")
md = notebook_summary["merged_dataset"]
print(f"  Rows              : {md['rows']:,}")
print(f"  Columns           : {md['columns']}")
print(f"  Stations          : {md['unique_stations']}")
print(f"  Weather matched   : {md['weather_matched_rows']:,}")
print(f"  Mean delay        : {md['delay_mean_min']} min")
print()

print("--- VALIDATION ---")
val = notebook_summary["merge_validation"]
print(f"  Merge validation  : {'PASSED' if val['passed'] else 'FAILED'}")
if val["failed_checks"]:
    print(f"  Failed checks     : {val['failed_checks']}")
print()

print("--- SAVED FILES ---")
for name, info in artifact_status.items():
    print(f"  {info['path']}  ({info['size_mb']} MB)")
print(f"  {SUMMARY_PATH}")
print()

print("=" * 72)
print("NOTEBOOK 04 COMPLETE — Ready for Notebook 05")
print("=" * 72)
print()
print("Key Takeaways:")
print(f"  • {md['rows']:,} ICE stops merged with weather features")
print(f"  • {geo['successful']} stations geocoded | {wx['unique_stations']} weather profiles")
print(f"  • Merge rate {mg['mergeable_merge_rate_pct']}% on mergeable rows")
print()
print("What Comes Next:")
print("  → Notebook 05, Section 1: EDA on ice_weather_merged_2024-07.parquet")
print(sep)

--- NOTEBOOK 04 ARTIFACT CHECK ---
  [OK] station_coordinates (0.03 MB)
  [OK] geocoding_report (0.05 MB)
  [OK] weather_by_station (0.3 MB)
  [OK] weather_fetch_report (0.02 MB)
  [OK] ice_weather_merged (6.21 MB)
  [OK] merge_validation_report (0.0 MB)

Notebook 04 — Integration Summary

--- PIPELINE ---
  Section 1 — Geocoded  : 95 stations  (100.0%)
  Section 2 — Weather   : 70,680 rows  (95 stations)
  Section 3 — Merged    : 146,389 rows  (merge rate 100.0%)

--- MERGED DATASET ---
  Rows              : 146,389
  Columns           : 32
  Stations          : 95
  Weather matched   : 132,268
  Mean delay        : 10.92 min

--- VALIDATION ---
  Merge validation  : PASSED

--- SAVED FILES ---
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\station_coordinates.json  (0.03 MB)
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\geocoding_report_2024-07.json  (0.05 MB)
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\weather_by_station_2024